# Showing Unicode Characters in a Table

In the lecture, we have defined the *Unicode alphabet* as the set
$$ \Sigma_{\textrm{Unicode}} = \{ 0, 1, \cdots, 1\,114\,111 \}, $$
i.e. every character is identified with a natural number, its *code point*.  Code points are usually
written in hexadecimal notation using the prefix `U+`, so the largest code point is `U+10FFFF`.

This notebook implements a small program that displays a range of Unicode characters in a table.  For
every character, the table shows both the *glyph*, i.e. the picture of the character, and its code point.
The official Unicode name of a character is shown when the mouse pointer rests on its glyph.

## Imports

- The module `unicodedata` gives access to the *Unicode Character Database*.  We use it to look up the
  name and the *general category* of a character.
- The module `html` provides the function `html.escape`, which we need since characters like `<` and `&`
  have a special meaning in HTML.
- The function `display` and the class `HTML` from the module `IPython.display` enable us to render an
  HTML string as a table inside the notebook.

In [ ]:
import unicodedata
import html
from IPython.display import display, HTML

## Printable Characters

Not every code point can be displayed:
- Many code points are not yet assigned to a character.
- Some code points are *control codes*, e.g. the code point $10$ encodes a line feed.
- Other code points are reserved for special purposes, e.g. the *surrogates* in the range from `U+D800`
  to `U+DFFF`, which are used by the encoding UTF-16.

Every character belongs to a *general category*, which is a string consisting of two letters.  The first
letter gives the major class of the character, e.g. `L` for letters, `N` for numbers, `P` for
punctuation, and `C` for *other* characters.  All code points that cannot be displayed belong to one of the
categories `Cc` (control), `Cf` (format), `Cs` (surrogate), `Co` (private use), and `Cn` (unassigned).
Besides these, the space characters of category `Zs` are displayed as nothing.

The function `is_printable(c)` checks whether the code point `c` stands for a character that has a visible
glyph.  The function `chr(c)` converts the code point `c` into a string of length $1$, while the function
`unicodedata.category` returns the general category of this character.

In [ ]:
def is_printable(c):
    category = unicodedata.category(chr(c))
    return category[0] != 'C' and category != 'Zs'

Let's test this function with the code points of the letter `A`, the space character, and the line feed.

In [ ]:
is_printable(65), is_printable(32), is_printable(10)

## Formatting a Single Cell

The function `cell(c)` returns the HTML code of the two table cells that describe the code point `c`.
- The first cell contains the glyph.  The glyph is escaped with `html.escape`, and the official name of
  the character, which is returned by `unicodedata.name`, is attached as the attribute `title`.  Browsers
  show this attribute as a *tooltip*.  If the character is not printable, the cell is left empty and gets a
  grey background instead.  We do not use a placeholder symbol since every symbol we could choose is itself a
  Unicode character and might therefore occur in the table.  The second argument of `unicodedata.name` is returned when a character has no name, which is
  the case, e.g., for control codes.
- The second cell contains the code point in the format `U+XXXX`.  The format specification `04X` writes
  the number `c` in hexadecimal notation with capital letters, using at least $4$ digits.

In [ ]:
def cell(c):
    name = unicodedata.name(chr(c), 'no name')
    if is_printable(c):
        glyph, style = html.escape(chr(c)), ''
    else:
        glyph, style = '', 'background:lightgrey; '
    return (f'<td title="{html.escape(name)}" style="{style}font-size:150%; text-align:center">{glyph}</td>'
            f'<td style="font-family:monospace">U+{c:04X}</td>')

## Building the Table

The function `unicode_table(first, last, columns)` returns an HTML table that shows all characters with code
points from `first` up to and including `last`.  Every row of the table shows `columns` characters.
- `codes` is the list of all code points that are to be shown.
- The header of the table contains the column titles `Glyph` and `Code` once for every character in a row.
- The list comprehension that computes `rows` splits the list `codes` into slices of length `columns`.
  Every slice is turned into one row of the table.  The last row might be shorter than the other rows.

In [ ]:
def unicode_table(first, last, columns=8):
    codes  = list(range(first, last + 1))
    header = '<tr>' + '<th>Glyph</th><th>Code</th>' * columns + '</tr>'
    rows   = [ '<tr>' + ''.join(cell(c) for c in codes[i:i+columns]) + '</tr>'
               for i in range(0, len(codes), columns)
             ]
    return HTML('<table>' + header + ''.join(rows) + '</table>')

## Trying it Out

We start with the printable characters of the *ASCII alphabet*, which are the characters with the code
points from $32$ up to $126$.  Compare this table with Table 1.1 of the lecture notes.

In [ ]:
display(unicode_table(32, 126))

The Greek letters start at the code point `U+0391`.  Note that there is no character with the code point
`U+03A2`: this code point is not assigned, so its cell is grey.  The lower case letter `ς` at `U+03C2` is the variant of `σ` that
is used at the end of a word, so there is no need for a corresponding capital letter.

In [ ]:
display(unicode_table(0x0391, 0x03C9))

Unicode also contains many mathematical symbols.  Some of the most important ones are found in the block
*Mathematical Operators*, which starts at the code point `U+2200`.

In [ ]:
display(unicode_table(0x2200, 0x225F))

The *CJK Unified Ideographs* are Chinese characters, which are also used in Japanese and Korean.  This block
starts at `U+4E00`.  The first of these characters is `一`, the Chinese character for the number $1$.

In [ ]:
display(unicode_table(0x4E00, 0x4E3F))

Finally, Unicode contains a large number of *emojis*.  The block *Emoticons* starts at `U+1F600`.  Note
that these code points are bigger than $2^{16} = 65\,536$, so they need more than $4$ hexadecimal digits.

In [ ]:
display(unicode_table(0x1F600, 0x1F64F))